## Agentic PDF Research Assistant — Walkthrough

This notebook documents the reasoning and step-by-step build process behind the project, from raw PDF text to a full agentic RAG pipeline built with **LangChain**, **FAISS**, and **LangGraph**.

The deployed app (`app.py` + `graph.py` + `ingestion.py`) is the deployed version of everything shown here — this notebook is for a walkthrough of working of the application with some explanation of each step.

**Pipeline:** `PDF → Chunking → Embeddings → FAISS → Classify → Retrieve/Summarize → Analyze → Generate`

### Setup

Before running, make sure you have:
1. A `.env` file in this folder with `GROQ_API_KEY=your_key_here` (you can get one at [console.groq.com](https://console.groq.com)), if you don't wanna go through hassle you can check the deployed version to see how it works anyway...
2. A sample PDF in this same folder — update `PDF_PATH` below to the path of pdf, or if in the same folder, with same name.
3. Installed dependencies: `pip install -r requirements.txt`


In [22]:
# If running for the first time, run this to install the requried modules
# %pip install langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq langgraph faiss-cpu sentence-transformers pypdf python-dotenv

from dotenv import load_dotenv
print(load_dotenv())

import os
NOTEBOOK_DIR = os.getcwd()  # This is done to avoid directory problems while running the notebook

PDF_PATH = os.path.join(NOTEBOOK_DIR, "sample.pdf")
# PDF_PATH = "sample.pdf"


True


### Step 1 — Loading a PDF

`PyPDFLoader` reads a PDF and returns one `Document` object **per page** — not one giant blob of text. Each `Document` has:
- `page_content`: the raw text of that page
- `metadata`: extra info, including which page number it came from (useful later for citing sources)

*since this pipeline only looks for text, the images in the document may not contribute to the document objects* 


In [23]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Number of pages loaded: {len(pages)}")
print("\n--- First page preview ---")
print(pages[0].page_content[:300])
print("\n--- Metadata of first page ---")
print(pages[0].metadata)


Number of pages loaded: 11

--- First page preview ---
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aid

--- Metadata of first page ---
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with r

### Step 2 — Chunking

Sometimes the pages are too large and have too much text to embed and search effectively as a single object. A page with too much of text prodcues a "blurry" embedding that doesn't represent any one idea well, the retrieval gets less precise.

`RecursiveCharacterTextSplitter` breaks documents into smaller overlapping chunks:
- `chunk_size`: max characters per chunk
- `chunk_overlap`: characters shared between consecutive chunks, so we don't lose context at the boundary when a sentence gets cut in half

We should expect **more chunks than pages**, since each page typically splits into several chunks. With the `sample.pdf` I used here, it produces 76 chunks for 11 pages in the pdf.


In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"Pages: {len(pages)}  |  Chunks: {len(chunks)}")
print("\n--- First chunk ---")
print(chunks[0].page_content)
print("\n--- Metadata (note the page number carries over) ---")
print(chunks[0].metadata)


Pages: 11  |  Chunks: 76

--- First chunk ---
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or

--- Metadata (note the page number carries over) ---
{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configura

### Step 3 — Embeddings

An embedding model converts text into a fixed-length vector of numbers such that texts with similar **meaning** end up close together in that vector space — regardless of exact wording.
The model used here i.e. `all-MiniLM-L6-v2` produces a vector of size `384` irrespective of length of text passed.


In [25]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_vector = embeddings.embed_query(chunks[0].page_content)
print(f"Vector length: {len(sample_vector)}")
print(f"First 5 values: {sample_vector[:5]}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12349.94it/s]


Vector length: 384
First 5 values: [-0.0664469450712204, -0.10133861750364304, 0.06421992182731628, -0.027600936591625214, 0.04505106061697006]


**Why 384 specifically?** Because that's the output size the embedding model (all-MiniLM-L6-v2) was trained to produce, a different embedding model might produce a vector of different dimension. It depends on the architecture of the model. Any length of text would produce a vector of 384 numbers but the difference is mainly in the position of that point in the 384 dimensional space.

### Step 4 — Building a FAISS index

FAISS (Facebook AI Similarity Search) is a library for fast nearest-neighbor search over vectors. We embed **every** chunk and store all the vectors in an index that can quickly answer: "which stored vectors are closest to this new query vector?"

We save the index to disk so it doesn't need to be rebuilt (i.e., re-embedded) every time we want to use it.


In [26]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print(f"Total vectors stored: {vectorstore.index.ntotal}")


Total vectors stored: 76


### Step 5 — Testing retrieval

When we ask a plain-English question, and FAISS finds the chunks whose *meaning* is closest to it — no keyword matching involved.

You can try giving it two queries, one clearly related to your PDF's content, and one totally unrelated. FAISS always returns its top-k nearest chunks, even for the unrelated query, it has no built-in concept of determining if it is just not relevant and return nothing. That limitation is exactly why we added the relevance check node ourselves, to filter out the unrelated query and make the LLM reply with a standard output of "I don't know".


In [27]:
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,
)

query = "What is this document about?"  # try changing this query
results = loaded_vectorstore.similarity_search(query, k=3) # k=3 is for top-k parameter

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()


--- Result 1 (page 1) ---
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension

--- Result 2 (page 10) ---
[21] Minh-Thang Luong, Hieu Pham, and Christopher D Manning. Effective approaches to attention-
based neural machine translation. arXiv preprint arXiv:1508.04025, 2015.
[22] Ankur Parikh, Oscar Täckström, Dipanjan Das, and Jakob Uszkoreit. A decomposable attention
model. In Empirical Methods in Natu

--- Result 3 (page 0) ---
our research.
†Work performed while at Google Brain.
‡Work performed while at Google Research.
31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.



In [28]:
# Now try an unrelated query
unrelated_query = "What is size of NIT Silchar?"
results = loaded_vectorstore.similarity_search(unrelated_query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()


--- Result 1 (page 2) ---
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1 Scaled Dot-Product Attention
We call our particular attent

--- Result 2 (page 5) ---
Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types.n is the sequence length,d is the representation dimension,k is the kernel
siz

--- Result 3 (page 2) ---
layers, produce outputs of dimensiondmodel = 512.
Decoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder insert



### Step 6 — Generating an answer (plain RAG, no agent yet)

Now we hand the retrieved chunks to an LLM as context and ask it to answer using **only** that context. This is the core idea of RAG: ground the answer in retrieved text instead of the model's general training knowledge, which reduces hallucination and lets us cite sources.

We use **Groq** to run the LLM — it's free and fast, and needs an API key from [console.groq.com](https://console.groq.com) stored in `.env` as `GROQ_API_KEY`. You must have added one while setting the app up after the pull.


In [29]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a research assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}
""")

def ask(question, k=3):
    docs = loaded_vectorstore.similarity_search(question, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    return response.content, docs

answer, docs = ask("What is attention?")  # try changing this query
print(answer)


Attention is a mechanism that maps a **query** and a set of **key‑value** pairs to an **output**.  
All of the query, keys, values, and the output are vectors, and the output is computed as a weighted sum of the values, where the weights are derived from the compatibility between the query and each key.  

In the context of language models, self‑attention (or intra‑attention) is a special case of this mechanism that relates different positions within a single sequence to compute a representation of that sequence.


In [30]:
# Confirm the grounding behavior: an unrelated question should get an honest "I don't know"
answer, docs = ask("What is the capital of France?")
print(answer)


I don't know.


At this point we have a **complete, working RAG pipeline** — but it's a straight line: retrieve → generate, no branching, no decision-making. Whatever gets retrieved gets fed straight into generation, and we're relying entirely on the prompt's wording ("say you don't know") to avoid a bad answer. That's fragile.

The rest of this notebook turns this into an **agent** using LangGraph — a small state machine that can make decisions about *how* to answer, not just generate text from whatever it's given.

### Step 7 — LangGraph concepts

Three ideas make up the whole system:

- **State** — a shared dictionary that flows through the pipeline (like a clipboard passed between steps). It starts with just a `question` and accumulates more fields (`docs`, `answer`, etc.) as it moves along.
- **Node** — a plain Python function: state in, state out. Each node does one job (retrieve, check relevance, generate...).
- **Edge / Conditional edge** — wiring between nodes. A normal edge is a fixed "then do this next." A **conditional edge** is a function that looks at the state and decides *which* node runs next — this is what makes the pipeline "agentic" rather than a fixed line.

### Step 8 — Defining the state


In [31]:
from typing import TypedDict, List
from langchain_core.documents import Document

class AgentState(TypedDict):
    question: str          # the user's original question
    search_query: str       # question rewritten to be without pronouns(also done using LLM)
    chat_history: str        # recent conversation, for follow-up context
    question_type: str        # "broad" or "specific", decided by the classify node
    docs: List[Document]        # chunks retrieved for this question
    is_relevant: bool             # whether check_relevance judged the docs useful
    answer: str                     # the final answer


### Step 9 — The nodes

Each node below is a small, single-purpose function. Read them in order — this is the same order data flows through the graph.

**`classify_question_node`** — decides if a question is asking about the *whole document* ("what is this paper about?") or *one specific thing* ("what is self-attention?"). This matters because similarity search (retrieval) is built for finding specific facts, not for summarizing an entire document — a "broad" question needs a different strategy entirely.


In [32]:
classify_prompt = ChatPromptTemplate.from_template("""
Given the recent conversation, classify the CURRENT question below into exactly one category:

- "broad": the question asks about the ENTIRE document as a whole — its overall
topic, purpose, main contributions, or a general summary. It cannot be answered
by looking at just one small section.
Examples: "What is this paper about?", "Summarize this document",
"What are the main contributions?"

- "specific": the question asks about ONE particular concept, term, fact, method,
number, or section — even if phrased vaguely or with pronouns like "it" or "that"
that refer back to something specific mentioned earlier in the conversation.
Examples: "What is self-attention?", "Why is it useful?" (when "it" refers to a
specific concept discussed earlier), "What optimizer was used?"

Recent conversation:
{chat_history}

Current question: {question}

Reply with ONLY one word: "broad" or "specific".
""")

def classify_question_node(state: AgentState) -> AgentState:
    chain = classify_prompt | llm
    response = chain.invoke({
        "question": state["question"],
        "chat_history": state.get("chat_history", ""),
    })
    verdict = str(response.content).strip().lower()
    state["question_type"] = "broad" if "broad" in verdict else "specific"
    return state


**Why `chat_history` matters here:** a follow-up like *"Why is it useful?"* looks vague in isolation, and an LLM with no context might misclassify it as "broad." Passing recent conversation lets the classifier see that "it" refers to something specific discussed a moment ago.

**`retrieve_node`** — for "specific" questions. Before searching, it **rewrites** the question into a standalone form using chat history (so "why is it useful?" becomes "why is self-attention useful?"), since FAISS has no memory of the conversation — it only sees whatever text string it's given.


In [33]:
rewrite_prompt = ChatPromptTemplate.from_template("""
Given the recent conversation and a follow-up question, rewrite the follow-up
into a standalone question that makes sense without needing the conversation.
If the question is already standalone, just repeat it unchanged.

Recent conversation:
{chat_history}

Follow-up question: {question}

Standalone question:
""")

def retrieve_node(state: AgentState) -> AgentState:
    chat_history = state.get("chat_history", "")
    if chat_history.strip():
        chain = rewrite_prompt | llm
        response = chain.invoke({"chat_history": chat_history, "question": state["question"]})
        search_query = str(response.content).strip()
    else:
        search_query = state["question"]  # nothing to rewrite for the first question

    state["search_query"] = search_query
    docs = loaded_vectorstore.similarity_search(search_query, k=5)
    state["docs"] = docs
    return state


**`check_relevance_node`** — the fix for FAISS's "always returns something" limitation. Instead of trusting whatever got retrieved, we explicitly ask the LLM: *does this content actually answer the question?* Only if the answer is yes do we proceed to generate a real answer.


In [34]:
relevance_prompt = ChatPromptTemplate.from_template("""
Look at the context and the question below.
Does the context contain information that could answer the question?
Reply with ONLY one word: "yes" or "no".

Context:
{context}

Question: {question}
""")

def check_relevance_node(state: AgentState) -> AgentState:
    context = "\n\n".join(doc.page_content for doc in state["docs"])
    chain = relevance_prompt | llm
    response = chain.invoke({
        "context": context,
        "question": state.get("search_query", state["question"]),
    })
    verdict = str(response.content).strip().lower()
    state["is_relevant"] = "yes" in verdict
    return state


**`generate_node`** and **`no_context_response_node`** — the two possible endings for a "specific" question, depending on the relevance verdict.

Note that `no_context_response_node` explicitly clears `state["docs"] = []`. Without this, stale (irrelevant) docs from the failed retrieval would still be sitting on the state and could incorrectly get displayed as "sources" for an answer that wasn't actually based on them.


In [35]:
rag_prompt = ChatPromptTemplate.from_template("""
You are a research assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.
Use the recent conversation only to understand what the question is referring to
(e.g. pronouns like "it" or "that") — the actual answer must still come from the context.

Recent conversation:
{chat_history}

Context:
{context}

Question: {question}
""")

def generate_node(state: AgentState) -> AgentState:
    context = "\n\n".join(doc.page_content for doc in state["docs"])
    chain = rag_prompt | llm
    response = chain.invoke({
        "context": context,
        "question": state["question"],
        "chat_history": state.get("chat_history", ""),
    })
    state["answer"] = str(response.content)
    return state

def no_context_response_node(state: AgentState) -> AgentState:
    state["answer"] = "I couldn't find relevant information in the document to answer that question."
    state["docs"] = []  # clear stale docs — they were judged irrelevant, don't show them as "sources"
    return state


**`summarize_node`** — the path for "broad" questions. Similarity search doesn't work well here (a query like "summarize this paper" has no specific topic to match against), so instead we feed the LLM a large slice of the document directly.

We cap it at 40 chunks to stay under Groq's free-tier rate limit (tokens-per-minute) — a known trade-off: summaries of very long documents may not reflect later sections. A more complete fix would use **map-reduce summarization** (summarize groups of chunks, then summarize those summaries), which avoids ever exceeding the limit — a good future improvement, left out here for simplicity.


In [36]:
summarize_prompt = ChatPromptTemplate.from_template("""
You are a research assistant. Use the full document content below to answer
the question, which is asking for an overview or summary.

Document:
{full_text}

Question: {question}
""")

def summarize_node(state: AgentState) -> AgentState:
    max_chunks = 40
    limited_chunks = chunks[:max_chunks]
    full_text = "\n\n".join(chunk.page_content for chunk in limited_chunks)
    chain = summarize_prompt | llm
    response = chain.invoke({"full_text": full_text, "question": state["question"]})
    state["answer"] = str(response.content)
    return state


### Step 10 — Conditional routing functions

These are the actual "decisions" in the graph — plain functions that read the state and return the **name** of whichever node should run next.


In [37]:
def route_after_classify(state: AgentState) -> str:
    return "summarize" if state["question_type"] == "broad" else "retrieve"

def route_after_relevance_check(state: AgentState) -> str:
    return "generate" if state["is_relevant"] else "no_context_response"


### Step 11 — Assembling the graph

Putting it all together: register every node under a name, wire the fixed edges, wire the conditional edges, then compile.

```
classify ──broad──► summarize ──► END
   │
   └──specific──► retrieve ──► check_relevance ──relevant──► generate ──► END
                                      │
                                      └──not relevant──► no_context_response ──► END
```


In [38]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("classify", classify_question_node)
graph.add_node("summarize", summarize_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("check_relevance", check_relevance_node)
graph.add_node("no_context_response", no_context_response_node)
graph.add_node("generate", generate_node)

graph.set_entry_point("classify")

graph.add_conditional_edges(
    "classify",
    route_after_classify,
    {"summarize": "summarize", "retrieve": "retrieve"},
)

graph.add_edge("retrieve", "check_relevance")

graph.add_conditional_edges(
    "check_relevance",
    route_after_relevance_check,
    {"generate": "generate", "no_context_response": "no_context_response"},
)

graph.add_edge("generate", END)
graph.add_edge("no_context_response", END)
graph.add_edge("summarize", END)

app = graph.compile()
print("Graph compiled successfully.")


Graph compiled successfully.


### Step 12 — Running the agent end to end

`app.invoke(...)` drops a fresh state in with just `question` (and optionally `chat_history`) filled in, and lets it flow through the whole graph until it reaches `END`.

Try all four categories below and compare the routing behavior:
1. A **specific** question the document can answer
2. A **broad** "what is this about" question
3. An **irrelevant** question (should hit the fallback)
4. A **follow-up** using a pronoun, to test chat-history-aware classification + query rewriting


In [39]:
result = app.invoke({"question": "What is self-attention?", "chat_history": ""}) # type: ignore
print("Type:", result["question_type"])
print("Answer:", result["answer"])


Type: specific
Answer: Self‑attention (also called intra‑attention) is an attention mechanism that relates different positions within a single sequence in order to compute a representation of that sequence. It maps a query and a set of key‑value pairs (all vectors) to an output, which is a weighted sum of the values.


In [40]:
result = app.invoke({"question": "What is this paper about?", "chat_history": ""}) # type: ignore
print("Type:", result["question_type"])
print("Answer:", result["answer"])


Type: broad
Answer: **“Attention Is All You Need” – Overview**

The paper introduces the **Transformer**, a new neural network architecture for sequence‑to‑sequence tasks that relies **entirely on attention mechanisms** and discards recurrent and convolutional layers. Its key points are:

| Aspect | What the paper does |
|--------|---------------------|
| **Motivation** | Recurrent models (RNNs, LSTMs, GRUs) compute hidden states sequentially, limiting parallelism and making training slow, especially for long sequences. Convolutional models reduce this but still require many layers to capture long‑range dependencies. |
| **Core Idea** | Replace all recurrence and convolution with **self‑attention** (scaled dot‑product attention) and **multi‑head attention**. This allows every token to attend to every other token in a single operation, giving constant‑time dependency paths. |
| **Architecture** | An encoder‑decoder stack (6 layers each). Each encoder layer has: 1) multi‑head self‑attent

In [41]:
result = app.invoke({"question": "What is the capital of France?", "chat_history": ""}) # type: ignore
print("Type:", result["question_type"])
print("Relevant:", result.get("is_relevant"))
print("Answer:", result["answer"])


Type: specific
Relevant: False
Answer: I couldn't find relevant information in the document to answer that question.


In [42]:
# Checking for pronouns
history = "User: What is self-attention?\nAssistant: Self-attention relates different positions within a single sequence to compute a representation of that sequence."

result = app.invoke({"question": "Why is it useful?", "chat_history": history}) # type: ignore
print("Type:", result["question_type"])
print("Rewritten query:", result.get("search_query"))
print("Answer:", result["answer"])


Type: specific
Rewritten query: Why is self‑attention useful?
Answer: Self‑attention is useful because it provides a flexible, efficient way to build a representation of a sequence by directly relating every position to every other position.  This mechanism has been shown to work well on a wide range of natural‑language tasks—reading comprehension, abstractive summarization, textual entailment, and learning task‑independent sentence representations.  In addition, self‑attention layers are computationally faster than recurrent layers when the sequence length is smaller than the hidden dimensionality, which is typical for sentence‑level models in machine translation.  Finally, the attention weights are interpretable: different heads often learn distinct syntactic or semantic roles, giving insight into the model’s reasoning.


**This concludes the working of the pipeline with a basic pipeline at first and then with structured flow with different nodes and edges to check different categories of queries implemented in LangGraph**